# Session 4: a fourth-order symplectic solver
**A. Real-world problem.** In Session 3 the implicit midpoint rule kept the energy error of the
double pendulum bounded, but only at the price of a *large* error at a coarse step: with
h = 0.02 s it wanders by 1.6 percent of the energy scale. Satellite orbits, molecules and planets
are simulated for billions of steps, so people want a solver that is both symplectic **and**
accurate. Today we build one by composing three midpoint steps, check that it really is
fourth order, count what it costs, and find out where it stops working.

Equal balls (1 kg), equal 1 m rods, no friction. The solver ideas here are general; the numbers
are for this pendulum released from rest with both rods at the same angle.

In [ ]:
import math
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81                 # gravity, m/s^2
E_REF = 3 * G            # energy scale (m1 + m2) g l1 + m2 g l2 = 29.43 J
CALLS = [0]              # how many times the Hamiltonian rate function has been evaluated (our measure of work)
NOT_CONVERGED = [0]      # how many midpoint solves gave up after max_iter tries

## Step 1: the pieces from Sessions 2 and 3
The equations of motion and RK4 (the reference), and the implicit midpoint rule on the
Hamiltonian form. Two small additions: every evaluation of the rate is counted in `CALLS`, so
we can measure *work* instead of guessing it, and a midpoint solve that fails to converge is
counted in `NOT_CONVERGED` instead of failing silently.

In [ ]:
def rate(state):
    """(theta1', theta2', omega1', omega2') for state (theta1, theta2, omega1, omega2)."""
    t1, t2, w1, w2 = state
    d = t1 - t2
    den = 3.0 - math.cos(2 * d)
    a1 = (-3 * G * math.sin(t1) - G * math.sin(t1 - 2 * t2) - 2 * math.sin(d) * (w2 ** 2 + w1 ** 2 * math.cos(d))) / den
    a2 = (2 * math.sin(d) * (2 * w1 ** 2 + 2 * G * math.cos(t1) + w2 ** 2 * math.cos(d))) / den
    return (w1, w2, a1, a2)


def energy(state):
    """Total energy E = T + V in joules (pivot is the zero of height)."""
    t1, t2, w1, w2 = state
    return w1 ** 2 + 0.5 * w2 ** 2 + w1 * w2 * math.cos(t1 - t2) - 2 * G * math.cos(t1) - G * math.cos(t2)


def release(a_deg):
    a = math.radians(a_deg)
    return (a, a, 0.0, 0.0)


def rk4_step(state, h):
    """Runge-Kutta 4: four slopes per step (not symplectic)."""
    def shift(s, k, f):
        return tuple(a + f * b for a, b in zip(s, k))
    k1 = rate(state)
    k2 = rate(shift(state, k1, h / 2))
    k3 = rate(shift(state, k2, h / 2))
    k4 = rate(shift(state, k3, h))
    return tuple(a + h / 6 * (b + 2 * c + 2 * d + e) for a, b, c, d, e in zip(state, k1, k2, k3, k4))


def _to_momenta(state):
    t1, t2, w1, w2 = state
    c = math.cos(t1 - t2)
    return (t1, t2, 2 * w1 + c * w2, c * w1 + w2)


def _to_rates(z):
    q1, q2, p1, p2 = z
    c = math.cos(q1 - q2)
    det = 2.0 - c * c
    return (q1, q2, (p1 - c * p2) / det, (2 * p2 - c * p1) / det)


def _hamilton_rate(z):
    CALLS[0] += 1
    q1, q2, p1, p2 = z
    w1, w2 = _to_rates(z)[2:]
    sd = math.sin(q1 - q2)
    return (w1, w2, -w1 * w2 * sd - 2 * G * math.sin(q1), w1 * w2 * sd - G * math.sin(q2))


def midpoint_step(state, h, tol=1e-14, max_iter=60):
    """One implicit-midpoint step (symplectic, second order). h may be negative (a step backward)."""
    z = _to_momenta(state)
    y = tuple(a + h * b for a, b in zip(z, _hamilton_rate(z)))
    for _ in range(max_iter):
        mid = tuple((a + b) / 2 for a, b in zip(z, y))
        y_new = tuple(a + h * b for a, b in zip(z, _hamilton_rate(mid)))
        change = max(abs(a - b) for a, b in zip(y, y_new))
        y = y_new
        if change < tol:
            break
    else:
        if change > 1e-9:              # tol is at round-off level, so only a clearly unsettled solve counts as a failure
            NOT_CONVERGED[0] += 1      # the substitution did not settle: this step is not trustworthy
    return _to_rates(y)

## Step 2: composing three midpoint steps
The midpoint step is *symmetric* (stepping forward h and then back -h returns to the start), and
its error per step starts with an h^3 term. Take three steps of sizes gamma1 h, gamma0 h, gamma1 h
in a row (a "triple jump"). Two conditions decide the sizes:

    2 gamma1 + gamma0 = 1            (the three steps add up to one step h)
    2 gamma1^3 + gamma0^3 = 0        (the h^3 error terms cancel)

The solution is gamma1 = 1 / (2 - 2^(1/3)) = 1.3512 and gamma0 = -2^(1/3) / (2 - 2^(1/3)) = -1.7024.
So one step of the new solver goes forward 1.35 h, **backward** 1.70 h, then forward 1.35 h. A
composition of symplectic steps is symplectic, and the symmetric order keeps the method
reversible, so the energy error stays bounded, now with the error of a fourth-order method.

In [ ]:
GAMMA1 = 1.0 / (2.0 - 2.0 ** (1.0 / 3.0))
GAMMA0 = -(2.0 ** (1.0 / 3.0)) / (2.0 - 2.0 ** (1.0 / 3.0))
TRIPLE_JUMP = (GAMMA1, GAMMA0, GAMMA1)


def order_conditions(gammas):
    """(sum of the sizes, sum of their cubes): a triple jump needs (1, 0)."""
    return sum(gammas), sum(g ** 3 for g in gammas)


def composed_step(state, h, gammas=TRIPLE_JUMP):
    """One step of the fourth-order solver: a midpoint step for each size in gammas times h."""
    for g in gammas:
        state = midpoint_step(state, g * h)
    return state


def reversibility_error(step, state, h):
    """Largest change of the state after a step h followed by a step -h (zero for a reversible method)."""
    back = step(step(state, h), -h)
    return max(abs(a - b) for a, b in zip(back, state))

## Step 3: measuring the order
A method of order p has a global error that shrinks like h^p, so halving the step divides the
error by 2^p: 4 for the midpoint rule (p = 2), 16 for the composition (p = 4). We compare with a
very fine RK4 reference (h = 0.0005 s) after a short time T, starting from a gentle motion where
the error has not been amplified by chaos yet.

In [ ]:
START_GENTLE = (0.5, -0.3, 0.2, 0.1)


def run(step, state0, duration, h):
    s = tuple(float(v) for v in state0)
    for _ in range(int(round(duration / h))):
        s = step(s, h)
    return s


def reference_state(state0, duration, h_ref=0.0005):
    return run(rk4_step, state0, duration, h_ref)


def state_error(step, state0, duration, h, ref=None):
    """Largest difference of any of the four state numbers from the fine RK4 reference."""
    ref = reference_state(state0, duration) if ref is None else ref
    return float(np.max(np.abs(np.array(run(step, state0, duration, h)) - np.array(ref))))


def observed_orders(errors):
    """For errors measured at steps h, h/2, h/4, ...: log2 of each error ratio (the observed order)."""
    return [math.log2(a / b) for a, b in zip(errors[:-1], errors[1:])]

## Step 4: energy over a long run
This is the test the roadmap sets: 1000 s at h = 0.02 s for the 120 degree release. `drift_curve`
records the energy error (E - E0) / E_REF every `stride` steps, here every 2 s.

**A warning about chaos.** This pendulum is chaotic, so a change of 1e-13 rad in the start (the
size of round-off) gives a different orbit after about 30 s, and the *largest* energy error of
one run is a lucky-or-unlucky number: for the composition it ranges from about 0.005 to 0.012
over nearby starts. So we also use `typical_worst_error`: the median, over a handful of nearby
starts, of the largest energy error. That number does not depend on the computer's last bit.

In [ ]:
def drift_curve(step, state0, duration, h, stride=100):
    """Times and relative energy error (E - E0) / E_REF, sampled every `stride` steps."""
    n = int(round(duration / h))
    s = tuple(float(v) for v in state0)
    e0 = energy(s)
    ts, ds = [], []
    for i in range(1, n + 1):
        s = step(s, h)
        if i % stride == 0:
            ts.append(i * h)
            ds.append((energy(s) - e0) / E_REF)
    return np.array(ts), np.array(ds)


def nearby_starts(a_deg, n=5, spread=1e-9, seed=11):
    """n starts that differ from the release by random amounts of about `spread` in all four numbers."""
    rng = np.random.default_rng(seed)
    base = np.array(release(a_deg))
    return [tuple(float(v) for v in base + spread * rng.standard_normal(4)) for _ in range(n)]

## Step 5: what it costs
One step of the composition is three midpoint solves, and each solve repeats the substitution
until it settles. So the right comparison is not "same step" but **same work**: we count the rate
evaluations (`CALLS`) and ask how much work each solver needs to reach a given accuracy. We
measure two accuracies: the typical largest energy error over a chaotic run, and the error of
the state after 5 s compared with the fine reference.

In [ ]:
def worst_energy_error(step, state0, duration, h, every=1.0):
    """(largest |E - E0| / E_REF seen every `every` seconds, rate evaluations used). Failure gives (inf, work)."""
    CALLS[0] = 0
    s = tuple(float(v) for v in state0)
    e0 = energy(s)
    n = int(round(duration / h))
    k = max(1, int(round(every / h)))
    worst = 0.0
    try:
        for i in range(1, n + 1):
            s = step(s, h)
            if i % k == 0:
                e = abs(energy(s) - e0) / E_REF
                if not math.isfinite(e):
                    return math.inf, CALLS[0]
                worst = max(worst, e)
    except (ValueError, OverflowError):
        return math.inf, CALLS[0]
    return worst, CALLS[0]


def typical_worst_error(step, a_deg, duration, h, n=5):
    """(median over n nearby starts of the largest energy error, median work, solves that did not settle)."""
    errs, works = [], []
    NOT_CONVERGED[0] = 0
    for s in nearby_starts(a_deg, n):
        e, w = worst_energy_error(step, s, duration, h)
        errs.append(e)
        works.append(w)
    return float(np.median(errs)), int(np.median(works)), NOT_CONVERGED[0]


def trajectory_error_and_work(step, state0, duration, h, ref):
    CALLS[0] = 0
    err = state_error(step, state0, duration, h, ref)
    return err, CALLS[0]


def ladder_results(step, a_deg, duration, ladder, n=5):
    """List of (h, typical largest energy error, typical work, unsettled solves) for each step in `ladder`."""
    return [(h,) + typical_worst_error(step, a_deg, duration, h, n) for h in ladder]


def cheapest_for_tolerance(results, tol):
    """(work, h, error) of the cheapest entry of `ladder_results` with error at most `tol`, or None."""
    ok = [(w, h, e) for h, e, w, _ in results if e <= tol]
    return min(ok) if ok else None

## Step 6: where it stops working
The midpoint step is *implicit*: we solve it by repeating a substitution, and that only settles
when h times the fastest rate of change is smaller than about 2. The composition steps **backward**
by 1.70 h, so its middle solve is the hardest one. When the repetition does not settle, the step
is wrong, and the energy error jumps. `ladder_results` reports, for each step, the typical energy
error and the number of solves that gave up (`inf` means the run produced garbage or crashed).

## See it
Figure 1: the energy error of RK4, the midpoint rule and the composition over 1000 s. Figure 2:
the error against the step, with slopes 2 and 4. Figure 3: error against work. Figure 4: what
happens at coarse steps.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    from matplotlib.ticker import NullFormatter

    def tidy(ax):
        """Light grid, and no crowded minor tick labels on log axes."""
        ax.grid(alpha=0.3, which="both")
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())

    def xticks(ax, vals):
        ax.set_xticks(vals)
        ax.set_xticklabels([f"{v:g}" for v in vals])

    OUT.mkdir(exist_ok=True)
    BLUE, ORANGE, MAGENTA, GREY = "#0072b2", "#c86a00", "#d81b60", "#555555"
    s0 = release(120.0)
    print("triple jump sizes:", round(GAMMA1, 6), round(GAMMA0, 6), "conditions (1, 0):", order_conditions(TRIPLE_JUMP))
    print("reversibility (step h then -h), h = 0.05 from the gentle start:",
          {n: f"{reversibility_error(f, START_GENTLE, 0.05):.1e}" for n, f in (("rk4", rk4_step), ("midpoint", midpoint_step), ("composed", composed_step))})

    # --- Figure 1: energy over 1000 s at h = 0.02
    fig1, ax1 = plt.subplots(figsize=(8, 4))
    for name, f, col, ls in (("RK4", rk4_step, MAGENTA, "-"), ("midpoint (order 2)", midpoint_step, ORANGE, "--"), ("composition (order 4)", composed_step, BLUE, "-")):
        NOT_CONVERGED[0] = 0
        t, d = drift_curve(f, s0, 1000.0, 0.02)
        ax1.semilogy(t, np.maximum(np.abs(d), 1e-6), color=col, ls=ls, lw=1.6, label=name)
        print(f"  {name:24s} worst energy error {np.abs(d).max():.4f} of E_ref   solves that did not converge: {NOT_CONVERGED[0]}")
    ax1.axhline(1e-2, color=GREY, ls=":", lw=1.2)
    ax1.text(995, 1.15e-2, "roadmap limit 1e-2", color=GREY, fontsize=8, ha="right")
    ax1.set_ylim(1e-6, 0.5)
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("|E - E0| / E_ref")
    ax1.set_title("Energy error of three solvers, 120 degree release, h = 0.02 s (one run)")
    tidy(ax1)
    ax1.legend(fontsize=8, loc="lower right")
    fig1.tight_layout()
    fig1.savefig(OUT / "energy_1000s.png", dpi=120)
    print("largest energy error over 1000 s for 5 nearby starts (h = 0.02):")
    for name, f in (("RK4", rk4_step), ("midpoint", midpoint_step), ("composition", composed_step)):
        worsts = [float(np.abs(drift_curve(f, s, 1000.0, 0.02)[1]).max()) for s in nearby_starts(120.0, 5)]
        print(f"  {name:12s}", [round(w, 4) for w in worsts], "median", round(float(np.median(worsts)), 4))

    # --- Figure 2: order
    hs = [0.1, 0.05, 0.025, 0.0125]
    ref = reference_state(START_GENTLE, 0.4)
    e_mid = [state_error(midpoint_step, START_GENTLE, 0.4, h, ref) for h in hs]
    e_com = [state_error(composed_step, START_GENTLE, 0.4, h, ref) for h in hs]
    print("order test errors (midpoint):", [f"{e:.2e}" for e in e_mid], "observed orders", [round(o, 2) for o in observed_orders(e_mid)])
    print("order test errors (composed):", [f"{e:.2e}" for e in e_com], "observed orders", [round(o, 2) for o in observed_orders(e_com)])
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    ax2.loglog(hs, e_mid, "o--", color=ORANGE, label="midpoint (order 2)")
    ax2.loglog(hs, e_com, "s-", color=BLUE, label="composition (order 4)")
    xs = np.array([hs[0], hs[-1]])
    ax2.loglog(xs, 0.5 * e_mid[0] * (xs / hs[0]) ** 2, ":", color=GREY, label="slope 2 (shifted down)")
    ax2.loglog(xs, e_com[1] * (xs / hs[1]) ** 4 * 0.5, "-.", color=GREY, label="slope 4 (shifted down)")
    ax2.set_xlabel("step h (s)")
    ax2.set_ylabel("largest state error after 0.4 s")
    ax2.set_title("Halving h: error / 4 for order 2, error / 16 for order 4")
    tidy(ax2)
    xticks(ax2, [0.0125, 0.025, 0.05, 0.1])
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "order_test.png", dpi=120)

    # --- Figure 3: work and precision (typical energy error over 60 s; state error after 5 s)
    ladder_m = [0.03, 0.02, 0.015, 0.01, 0.0075, 0.005, 0.0035, 0.0025]
    ladder_c = [0.03, 0.025, 0.02, 0.015, 0.01, 0.0075, 0.005]
    ref5 = reference_state(s0, 5.0)
    res = {"midpoint": ladder_results(midpoint_step, 120.0, 60.0, ladder_m),
           "composition": ladder_results(composed_step, 120.0, 60.0, ladder_c)}
    tr = {}
    for name, f in (("midpoint", midpoint_step), ("composition", composed_step)):
        tr[name] = [trajectory_error_and_work(f, s0, 5.0, h, ref5) for h in [0.05, 0.025, 0.0125, 0.00625, 0.003125]]
        print(name, "energy (h, typical error, work, unsettled):", [(h, f"{e:.1e}", w, b) for h, e, w, b in res[name]])
        print(name, "state error after 5 s (error, work):", [(f"{e:.1e}", w) for e, w in tr[name]])
    fig3, axs3 = plt.subplots(1, 2, figsize=(9.6, 4.0))
    for name, col, mk in (("midpoint", ORANGE, "o--"), ("composition", BLUE, "s-")):
        good = [(w, max(e, 1e-7)) for _, e, w, _ in res[name] if math.isfinite(e)]
        axs3[0].loglog(*zip(*good), mk, color=col, label=name)
        axs3[1].loglog([w for e, w in tr[name]], [e for e, w in tr[name]], mk, color=col, label=name)
    axs3[0].set_xlabel("rate evaluations (work)")
    axs3[0].set_ylabel("typical largest |E - E0| / E_ref over 60 s")
    axs3[0].set_title("Energy error: the midpoint rule is cheaper for loose limits", fontsize=10)
    axs3[1].set_xlabel("rate evaluations (work)")
    axs3[1].set_ylabel("state error after 5 s")
    axs3[1].set_title("State error after 5 s: order 4 is far cheaper", fontsize=10)
    for a in axs3:
        tidy(a)
        a.legend(fontsize=8)
    xticks(axs3[0], [2e4, 5e4, 1e5, 2e5, 4e5])
    xticks(axs3[1], [3e3, 5e3, 1e4, 2e4, 4e4])
    fig3.tight_layout()
    fig3.savefig(OUT / "work_precision.png", dpi=120)
    for tol in (1e-2, 1e-3, 1e-4):
        print(f"tolerance {tol:g}: midpoint {cheapest_for_tolerance(res['midpoint'], tol)}   composition {cheapest_for_tolerance(res['composition'], tol)}")

    # --- Figure 4: coarse steps
    ladder = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.05]
    surveys = {n: ladder_results(f, 120.0, 60.0, ladder) for n, f in (("midpoint", midpoint_step), ("composition", composed_step))}
    fig4, ax4 = plt.subplots(figsize=(6.8, 4.2))
    for name, col, mk in (("midpoint", ORANGE, "o--"), ("composition", BLUE, "s-")):
        rows = surveys[name]
        ok = [(h, e) for h, e, _, _ in rows if math.isfinite(e)]
        ax4.loglog([h for h, _ in ok], [max(e, 1e-6) for _, e in ok], mk, color=col, label=name)
        bad = [h for h, e, _, _ in rows if not math.isfinite(e)]
        ax4.plot(bad, [3.0] * len(bad), "x", color=MAGENTA, ms=9, mew=2, label="solver failed" if name == "composition" else None)
        print(name, [(h, f"{e:.1e}", n) for h, e, _, n in rows])
    ax4.axhline(1.0, color=GREY, ls=":", lw=1)
    ax4.set_xlabel("step h (s)")
    ax4.set_ylabel("typical largest |E - E0| / E_ref over 60 s")
    ax4.set_title("The fourth-order solver gives up at a smaller step than the midpoint rule", fontsize=10)
    tidy(ax4)
    xticks(ax4, [0.005, 0.01, 0.02, 0.04])
    ax4.legend(fontsize=8)
    fig4.tight_layout()
    fig4.savefig(OUT / "step_limit.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Call `order_conditions` on `(0.5, 0.5)`, on `(1/3, 1/3, 1/3)` and on `TRIPLE_JUMP`. Which of them add up to 1, which have cubes that add up to 0, and what does that tell you about their order?
2. **Stretch.** Run `ladder_results(composed_step, 120.0, 60.0, [0.025, 0.03])` and `ladder_results(midpoint_step, 120.0, 60.0, [0.025, 0.03])`. Which solver has the larger typical energy error at each step, and how many solves did not converge?
3. **Explain.** Explain why a method that is more accurate per step can be the worse choice for a loose accuracy limit.

## Assessment
- **A. Conceptual.** A classmate says: "The fourth-order solver is simply better than the midpoint rule: it is just as symplectic and much more accurate." Give two reasons why this is too simple.
- **B. Numerical.** At h = 0.05 s the composition's state error after 0.4 s is 2.75e-4. Predict the error at h = 0.0125 s, and compare the number of midpoint solves per second of simulated time at the two steps. Give a sanity check.
- **C. Graph and simulation interpretation.** Figure 3 (left) gives, for the typical largest energy error over 60 s, 43,328 evaluations for the midpoint rule and 121,330 for the composition at 1e-2, and for 1e-4 only the composition (192,125). Which solver would you choose for each limit? Why does the composition's curve fall so much faster as the work grows?
- **D. Design and optimisation challenge.** Design: you need the typical largest energy error of the 120 degree release over 60 s to stay below 1e-4 of E_ref, with a budget of 200,000 rate evaluations. Choose a solver and a step, check your margin against the step at which the solver fails, and say what you would re-test if the pendulum were released from 150 degrees.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    s0 = L.release(120.0)

    # worked example: the triple-jump sizes by hand
    sizes = [g * 0.02 for g in L.TRIPLE_JUMP]
    checks.append((f"worked example: gamma1 = 1 / (2 - 2^(1/3)) = 1.3512 and gamma0 = -1.7024, so h = 0.02 s splits into {sizes[0]:.6f}, {sizes[1]:.6f}, {sizes[2]:.6f} s",
                   abs(L.GAMMA1 - 1.351207) < 1e-5 and abs(L.GAMMA0 + 1.702414) < 1e-5 and abs(sum(sizes) - 0.02) < 1e-15
                   and abs(sizes[0] - 0.027024) < 1e-6 and abs(sizes[1] + 0.034048) < 1e-6))
    one, cubes = L.order_conditions(L.TRIPLE_JUMP)
    naive = L.order_conditions((1 / 3, 1 / 3, 1 / 3))
    checks.append((f"order conditions: the sizes sum to {one:.1f} and their cubes to {cubes:.0e}; three equal thirds fail the cube test (cubes sum to {naive[1]:.3f})",
                   abs(one - 1.0) < 1e-12 and abs(cubes) < 1e-12 and abs(naive[1] - 1 / 9) < 1e-12))

    # symplectic and reversible: step h then -h returns to the start
    gentle = L.START_GENTLE
    rev = {n: L.reversibility_error(f, gentle, 0.05) for n, f in (("rk4", L.rk4_step), ("midpoint", L.midpoint_step), ("composed", L.composed_step))}
    checks.append((f"reversibility: step h then step -h returns to the start for the midpoint rule ({rev['midpoint']:.0e}) and the composition ({rev['composed']:.0e}) but not for RK4 ({rev['rk4']:.0e})",
                   rev["midpoint"] < 1e-12 and rev["composed"] < 1e-12 and rev["rk4"] > 1e-9))
    checks.append(("the composition changes nothing when the three sizes are 0 and 1 and 0 (a single midpoint step)",
                   max(abs(a - b) for a, b in zip(L.composed_step(gentle, 0.03, (0.0, 1.0, 0.0)), L.midpoint_step(gentle, 0.03))) < 1e-14))

    # order of accuracy
    hs = [0.1, 0.05, 0.025, 0.0125]
    ref = L.reference_state(gentle, 0.4)
    e_mid = [L.state_error(L.midpoint_step, gentle, 0.4, h, ref) for h in hs]
    e_com = [L.state_error(L.composed_step, gentle, 0.4, h, ref) for h in hs]
    o_mid, o_com = L.observed_orders(e_mid)[-1], L.observed_orders(e_com)[-1]
    checks.append((f"order: halving h from 0.025 to 0.0125 divides the midpoint error by {2 ** o_mid:.1f} (order {o_mid:.2f}) and the composition error by {2 ** o_com:.1f} (order {o_com:.2f})",
                   1.9 < o_mid < 2.1 and 3.8 < o_com < 4.2))
    checks.append((f"hand check (Section F): from {e_com[1]:.2e} at h = 0.05 an order-4 method predicts {e_com[1] / 256:.2e} at h = 0.0125; measured {e_com[3]:.2e}",
                   abs(e_com[3] / (e_com[1] / 256) - 1) < 0.15))

    # work counting
    L.CALLS[0] = 0
    s = gentle
    for _ in range(50):
        s = L.midpoint_step(s, 0.02)
    w_mid = L.CALLS[0]
    L.CALLS[0] = 0
    s = gentle
    for _ in range(50):
        s = L.composed_step(s, 0.02)
    w_com = L.CALLS[0]
    checks.append((f"work: one composed step costs {w_com / w_mid:.1f} times a midpoint step in rate evaluations (between 3 and 4.5)", 3.0 < w_com / w_mid < 4.5))

    # the roadmap test: 1000 s at h = 0.02, 120 degree release. Chaos makes one run's worst value luck,
    # so each solver is run from 5 nearby starts and the median is used.
    worst = {}
    L.NOT_CONVERGED[0] = 0
    for name, f in (("RK4", L.rk4_step), ("midpoint", L.midpoint_step), ("composition", L.composed_step)):
        worst[name] = float(np.median([np.abs(L.drift_curve(f, s, 1000.0, 0.02)[1]).max() for s in L.nearby_starts(120.0, 5)]))
    checks.append((f"1000 s at h = 0.02 s, 120 degrees (median of 5 nearby starts): worst energy error {worst['composition']:.4f} of E_ref for the composition, at most 0.01",
                   worst["composition"] <= 0.01))
    checks.append((f"for comparison the midpoint rule gives {worst['midpoint']:.4f} (above 0.01) and RK4 gives {worst['RK4']:.3f} (above 0.1)",
                   worst["midpoint"] > 0.01 and worst["RK4"] > 0.1))
    checks.append(("all solves converged in those runs (no silent failures)", L.NOT_CONVERGED[0] == 0))

    # failure case: coarse steps
    mid_bad = L.ladder_results(L.midpoint_step, 120.0, 60.0, [0.035])[0]
    com_bad = L.ladder_results(L.composed_step, 120.0, 60.0, [0.035])[0]
    checks.append((f"failure case: at h = 0.035 s the midpoint rule still runs (typical energy error {mid_bad[1]:.2f}) but the composition fails (error {com_bad[1]}, {com_bad[3]} solves did not converge)",
                   math.isfinite(mid_bad[1]) and mid_bad[1] < 0.3 and not math.isfinite(com_bad[1]) and com_bad[3] > 0))

    # cost and accuracy (typical largest energy error over 60 s; ladders are subsets of the ones in the lesson)
    res_m = L.ladder_results(L.midpoint_step, 120.0, 60.0, [0.015, 0.0075, 0.0025])
    res_c = L.ladder_results(L.composed_step, 120.0, 60.0, [0.02, 0.015, 0.01])
    m2, c2 = L.cheapest_for_tolerance(res_m, 1e-2), L.cheapest_for_tolerance(res_c, 1e-2)
    m3, c3 = L.cheapest_for_tolerance(res_m, 1e-3), L.cheapest_for_tolerance(res_c, 1e-3)
    m4, c4 = L.cheapest_for_tolerance(res_m, 1e-4), L.cheapest_for_tolerance(res_c, 1e-4)
    checks.append((f"mini-project: for an energy error of 1e-2 the midpoint rule is cheaper ({m2[0] if m2 else None} against {c2[0] if c2 else None} rate evaluations); the same at 1e-3 ({m3[0] if m3 else None} against {c3[0] if c3 else None})",
                   m2 is not None and c2 is not None and m3 is not None and c3 is not None and m2[0] < c2[0] and m3[0] < c3[0]))
    checks.append((f"mini-project: for 1e-4 only the composition reaches it on this ladder (h = {c4[1] if c4 else None}, {c4[0] if c4 else None} evaluations; the midpoint rule's best is {res_m[-1][1]:.1e})",
                   m4 is None and c4 is not None and c4[2] < 1e-4 and res_m[-1][1] > 5e-5))
    ref5 = L.reference_state(s0, 5.0)
    e_m, w_m = L.trajectory_error_and_work(L.midpoint_step, s0, 5.0, 0.003125, ref5)
    e_c, w_c = L.trajectory_error_and_work(L.composed_step, s0, 5.0, 0.0125, ref5)
    checks.append((f"mini-project: for the state after 5 s, similar work ({w_m} against {w_c} evaluations) gives an error {e_m:.1e} for the midpoint rule and {e_c:.1e} for the composition (at least 5 times smaller)",
                   w_c < 1.3 * w_m and e_c < e_m / 5))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
